# Naive Bayes, LDA, Random Forest and SVM

Four classifiers the other notebooks do not reach, compared on one binary
dataset so the differences are about the models rather than the data.

They make very different assumptions, and that is the point of putting them
side by side:

| model | what it assumes |
|---|---|
| Naive Bayes | features are conditionally independent given the class |
| LDA | two Gaussian classes sharing a covariance structure |
| Random Forest | nothing parametric; averages many decorrelated trees |
| SVM | a margin is what matters, via a kernel |


In [ ]:
import os
import numpy as np

from si.data import Dataset, summary
from si.util import train_test_split, accuracy_score, confusion_matrix
from si.util import CrossValidationScore
from si.supervised import NaiveBayes, LDA, RandomForest

## The data

`breast-bin` is the Wisconsin breast-cancer dataset: 9 integer-valued cell
measurements, and a binary malignant/benign label.


In [ ]:
DIR = os.path.dirname(os.path.realpath('.'))
dataset = Dataset.from_data(os.path.join(DIR, 'datasets/breast-bin.data'),
                           labeled=True)
print('X:', dataset.X.shape, ' y:', dataset.y.shape)
print('classes:', np.unique(dataset.y), ' counts:', np.bincount(dataset.y.astype(int)))
summary(dataset)

In [ ]:
np.random.seed(0)
train, test = train_test_split(dataset, split=0.7)
print(f'{len(train)} train, {len(test)} test')

## Naive Bayes

This implementation is the *categorical/multinomial* Naive Bayes: it models
each feature as a count, so it wants discrete inputs. Binarising each feature
against its median is the simplest way to get there.

"Naive" is the independence assumption -- it treats every feature as
independent of the others given the class, which is plainly false for cell
measurements that vary together. It works anyway, which is the interesting
part: the *ranking* of the class posteriors survives assumptions the
probabilities themselves do not.


In [ ]:
medians = np.median(train.X, axis=0)
train_bin = Dataset((train.X > medians).astype(int), train.y)
test_bin = Dataset((test.X > medians).astype(int), test.y)

nb = NaiveBayes()
nb.fit(train_bin)
print('train accuracy:', round(nb.cost(), 4))
print('test  accuracy:', round(accuracy_score(test_bin.y, nb.predict(test_bin.X)), 4))

In [ ]:
# Posteriors, not just labels: the class probabilities for a few samples.
probabilities = nb.predict_proba(test_bin.X[:5])
for row, true in zip(probabilities, test_bin.y[:5]):
    print(f'P(class) = {np.round(row, 3)}   true = {int(true)}')
print()
print('rows sum to 1:', np.allclose(probabilities.sum(axis=1), 1))

## Linear Discriminant Analysis

LDA finds the single direction that best separates the two classes -- it
maximises the distance between the class means relative to the spread within
each class. That makes it two things at once: a classifier, and a supervised
dimensionality reduction down to one number per sample.

Compare with PCA in eval1, which finds the direction of greatest variance
while ignoring the labels entirely. LDA uses them.


In [ ]:
lda = LDA()
lda.fit(train)
print('discriminant direction w:', np.round(lda.w, 4))
print('threshold:', round(lda.threshold, 4))
print()
print('train accuracy:', round(lda.cost(), 4))
print('test  accuracy:', round(accuracy_score(test.y, lda.predict(test.X)), 4))

In [ ]:
# As a transformer: every sample collapsed onto one discriminative number.
projected = lda.transform(train)
print('projected shape:', projected.shape)
for label in (0, 1):
    values = projected[train.y == label]
    print(f'class {label}: mean {values.mean():7.3f}  std {values.std():6.3f}')
print()
print('threshold sits between the two class means:',
      min(projected[train.y == 0].mean(), projected[train.y == 1].mean())
      < lda.threshold <
      max(projected[train.y == 0].mean(), projected[train.y == 1].mean()))

## Random Forest

A single decision tree can fit almost anything and generalises poorly. A forest
trains many trees, each on a bootstrap resample of the rows AND a random subset
of the columns, then takes a majority vote.

The double randomisation is what matters: it makes the trees *disagree*.
Averaging many correlated trees would help very little, since they would all be
wrong together.


In [ ]:
for n_estimators in (1, 5, 25, 100):
    np.random.seed(0)
    forest = RandomForest(n_estimators=n_estimators, max_depth=5)
    forest.fit(train)
    test_accuracy = accuracy_score(test.y, forest.predict(test.X))
    print(f'{n_estimators:4d} trees: train {forest.cost():.4f}   test {test_accuracy:.4f}')

## SVM

The SVM solves its dual as a quadratic program, which needs the optional
[cvxopt](https://cvxopt.org/) dependency (`pip install -e .[svm]`). It is also
the one model here that insists on a particular label encoding: the margin
term `y * (w.x + b)` only means anything for labels in {-1, +1}, so the {0, 1}
labels have to be remapped.

The cell below skips itself if cvxopt is absent rather than failing.


In [ ]:
try:
    from si.supervised.svm import SVM, linear_kernel, rbf_kernel
    from cvxopt import solvers
    solvers.options['show_progress'] = False   # the solver is chatty by default
    have_cvxopt = True
except ImportError:
    have_cvxopt = False
    print('cvxopt is not installed; skipping. pip install -e .[svm]')

In [ ]:
if have_cvxopt:
    # {0, 1} -> {-1, +1}: the dual and the sign() decision rule need signed labels.
    signed_train = Dataset(train.X, np.where(train.y == 0, -1.0, 1.0))
    signed_test = Dataset(test.X, np.where(test.y == 0, -1.0, 1.0))

    for name, kernel in (('linear', linear_kernel), ('rbf', rbf_kernel)):
        model = SVM(kernel=kernel, C=1)
        model.fit(signed_train)
        train_acc = model.cost()
        test_acc = accuracy_score(signed_test.y, model.predict(signed_test.X))
        print(f'{name:7} kernel: train {train_acc:.4f}   test {test_acc:.4f}'
              f'   support vectors: {len(model.lagr_multipliers)}')

In [ ]:
if have_cvxopt:
    # Passing the wrong encoding is refused rather than silently scoring ~0.5.
    try:
        SVM(kernel=linear_kernel).fit(train)   # still {0, 1}
    except ValueError as error:
        print('as expected:', error)

## Side by side

Cross-validated, so the comparison does not hinge on one lucky split. Note
Naive Bayes is scored on the binarised copy -- it is a different representation
of the same data, which is part of what the model brings.


In [ ]:
np.random.seed(0)
binarised = Dataset((dataset.X > np.median(dataset.X, axis=0)).astype(int),
                    dataset.y)

candidates = [
    ('Naive Bayes', NaiveBayes(), binarised),
    ('LDA', LDA(), dataset),
    ('Random Forest', RandomForest(n_estimators=25, max_depth=5), dataset),
]

for name, model, data in candidates:
    scores = CrossValidationScore(model, data, score=accuracy_score,
                                  cv=5, random_state=0).run()
    test_scores = np.array(scores[1])
    print(f'{name:14} 5-fold accuracy: {test_scores.mean():.4f} '
          f'+/- {test_scores.std():.4f}')

In [ ]:
# Where the errors fall matters more than how many there are: on a cancer
# screen a false negative and a false positive are not equivalent.
lda_model = LDA()
lda_model.fit(train)
confusion_matrix(test.y, lda_model.predict(test.X))

## Choosing between them

- **Naive Bayes** is the cheapest thing that works, trains in one pass, and
  needs discretised features here. Its independence assumption is wrong
  almost always and it remains a reasonable baseline.
- **LDA** gives an interpretable direction and doubles as a 1-D projection, but
  assumes two roughly Gaussian classes and only handles two of them.
- **Random Forest** asks least of the data and usually scores well without
  tuning, at the cost of interpretability and of training many trees.
- **SVM** is strong on small, clean datasets and lets the kernel do the
  non-linear work -- but it needs a QP solver, scales poorly with sample count,
  and is fussy about its label encoding.

Try next:

1. Swap in `hearts-bin.data` or `pima.data`; the ranking above is not fixed.
2. Vary `max_depth` in the forest and watch train and test accuracy diverge.
